# View synthesis responses from Qwen

Use the **extract** kernel. Run cells in order: list a paper's samples, choose a sample ID, load its full Markdown and send it to Qwen. Prompts are editable below. Nothing is saved; the database and papers are only read.

## The 20 papers

Sample counts include **all sample records in the database** for each paper, not just the one sample selected for the pilot. Counts read on 20 October 2026.

| Paper UID | No. of samples | Title |
|---|---:|---|
| ID001100849 | 5 | Physics Augmented Machine Learning Discovery of Composition-Dependent Constitutive Laws for 3D Printed Digital Materials |
| ID001098910 | 7 | Polycaprolactone/graphite nanoplates composite nanopapers |
| ID001098286 | 6 | Interfacial Entropic Interactions Tunes Fragility and Dynamic Heterogeneity of Glassy Athermal Polymer Nanocomposite films |
| ID001098538 | 4 | Magnetic Shape Memory Polymers with Integrated Multifunctional Shape Manipulations |
| ID000573272 | 10 | Continuous glycolysis of waste PET via reactive extrusion and copolymerization with PTMG for enhanced toughness |
| ID000415115 | 6 | Effect of chemically modified lignin addition on the physicochemical properties of PCL nanofibers |
| ID000588793 | 11 | Structure-property relationships and microstructural origins of elasticity in Poly(ethylene brassylate)-block-poly(tetramethylene oxide) multiblock copolymers and their fibers |
| ID000581823 | 8 | Thermal properties, rheology and foams of polystyrene-block-poly(4-vinylpyridine) diblock copolymers |
| ID001002844 | 20 | Dispersant and Protective Roles of Amphiphilic Poly(ethylene phosphate) Block Copolymers in Polyester/Bone Mineral Composites. |
| ID001032337 | 6 | Biobased Thermoplastic Elastomers: Structure-Property Relationship of Poly(hexamethylene 2,5-furanodicarboxylate)-Block-Poly(tetrahydrofuran) Copolymers Prepared by Melt Polycondensation. |
| ID000752848 | 5 | Trans crystallization behavior and strong reinforcement effect of cellulose nanocrystals on reinforced poly(butylene succinate) nanocomposites. |
| ID000180835 | 2 | A comparative investigation of uranium and thorium adsorption behavior on amidoximated copolymeric hydrogel |
| ID000180254 | 6 | Flexible Conductive Nanocomposites for Electrochemical Devices Based on Chlorinated Natural Rubber/Nickel Oxide Nanoparticles |
| ID000750264 | 2 | A surface-grafted thermoresponsive hydrogel in which the surface structure dominates the bulk properties. |
| ID000732480 | 6 | Enhancing the degradability of poly(lactic acid) through the introduction of main-chain thionoester linkages by ring-opening copolymerization |
| ID000732356 | 16 | Ring-opening polymerization of six-membered cyclic hybrid dimers composed of an oxoester and thioester |
| ID000905310 | 24 | Green revolution in polymer composites: performance and degradability of natural fiber reinforced biopolymers |
| ID000137005 | 7 | A study of oriented conductive composites with segregated network structure obtained via solid-state processing of UHMWPE reactor powder and carbon nanofillers |
| ID000143254 | 4 | Synthesis of waterborne acrylic copolymer resin as a binding agent for the development of water-based inks in the printing application |
| ID000061162 | 4 | Antibacterial and thermomechanical properties of composites of polylactic acid modified with capsicum oleoresin-impregnated nanoporous silica |

In [13]:
import sys
sys.dont_write_bytecode = True
import json
import re
import sqlite3
from pathlib import Path

import pandas as pd
from IPython.display import display
from openai import AsyncOpenAI

# Find the repository from either notebooks/ or the repository root.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "data/polymer_samples.db").is_file())

BASE_URL = "http://localhost:8000/v1"
MODEL = "qwen3.6-27b"
THINKING = True  # Set False for thinking off.


## Prompts

Edit these strings as needed. Keep the four placeholders in `USER_PROMPT` so the chosen sample record and full paper can be inserted.

In [ ]:
SYSTEM_PROMPT = """
# Task

Extract the complete synthesis and preparation procedure for **one requested polymer/material sample** from the supplied paper.

Use the database record **only to identify the target material**. All procedural information must come from the paper, including its Methods, Results, tables, captions, schemes, footnotes, and supplementary text.

Return exactly:

`{"procedure_text": "..."}`

The value must be a single paragraph of past-tense scientific prose.

## 1. Identify the material and its full synthesis lineage

Identify the exact sample and its corresponding formulation, experimental run, or table entry.

Trace its preparation backward through all required precursors, including synthesized monomers, initiators, fillers, prepolymers, and parent polymers.

Describe the **complete in-paper preparation of each precursor before its use**, followed by the preparation of the target material.

- Use the target sample's specific quantities and conditions, not those of another sample or a representative experiment.
- Expand shared synthesis procedures using the target sample's own parameters.
- Include preparation details wherever they appear in the paper, not just in the Methods section.
- Do not reconstruct procedures that are only referenced through external publications.

If the target cannot be identified or no preparation is available, return `{"procedure_text": ""}`.

## 2. Preparation Details and Source Fidelity

Describe the complete preparation chronologically, from starting materials to the final material. Include all reported:

- **Materials:** identities, quantities, ratios, concentrations, purity, grade, molecular weight, particle size, physical form, and other specifications.
- **Operations:** reagent pretreatment, synthesis, purification, work-up, modification, formulation, blending, curing, and material processing (e.g., casting, molding, hot-pressing, annealing).
- **Conditions:** addition order, solvents, catalysts, atmosphere, temperature, time, pressure, equipment, washing, drying, and storage.

Include shaping when it produces the material, but exclude preparation solely for characterization or testing.

**Preserve the paper's information exactly.** Never invent, infer, calculate, convert, or silently correct procedural details. Retain original units, ranges, approximations, inequalities, concentration bases, and ratios. Associate each quantity with its correct material and step.

When the paper gives a representative or typical procedure plus a table, take the operations from the procedure but the amounts, ratios and times from the sample's own table row. Omit the representative run's specific amounts when they differ from that row.

**Examples:**

- **Combined loading:** "5 wt% CNT/graphene (1:2)" → Retain the combined loading and ratio; do not calculate individual percentages.
- **No inference:** "Heated at 80 °C with a condenser" ≠ "Refluxed at 80 °C."
- **Include processing:** "Melt-mixed at 180 °C for 10 min and compression-molded at 190 °C" → Retain both operations and their conditions.

## 3. Exclude results and non-preparation information

Exclude:

- **Measured outcomes:** yield, conversion, selectivity, isolated product mass, measured molecular weight, dispersity, polymerization degree, substitution degree, actual composition, and measured material properties.
- **Appearance and morphology:** descriptions such as white powder, yellow oil, transparent, viscous, or homogeneous.
- **Sample identifiers:** target sample codes, parent sample codes, compound numbers and series labels (e.g., "P20", "monomer 1", "poly3", "PGT-1"). Refer to every material, including synthesized monomers, initiators, prepolymers and parent polymers, by the chemical name or description the paper gives (e.g., "3-methyl-1,4-oxathiane-2,5-dione", "hydroxyl-terminated BHET oligomer"). If the paper gives no chemical name, describe the material by how it was made (e.g., "the cyclized product", "the amine-functionalized silica"). Never construct a chemical name that the paper does not state.
- **Testing:** characterization methods, ageing/degradation experiments, measurement thermal cycles, and test-specific specimen preparation.
- **Missing-information:** never mention unavailable conditions or externally referenced procedures.

Retain actual input amounts used in subsequent steps and explicitly stated target/theoretical recipe parameters.

**Example — remove outcomes but retain conditions:**

Paper: "The reaction was stirred at 60 °C for 12 h, yielding 2.5 g of a yellow polymer with 95% conversion."
Recipe: "The reaction was stirred at 60 °C for 12 h."

**Example — distinguish measured results from input amounts:**

Paper: "The polymer was obtained in 85% yield. Subsequently, 1.0 g of the polymer was dissolved in 20 mL of DMF."
Recipe: "The polymer (1.0 g) was dissolved in 20 mL of DMF."

**Example — remove labels but retain formulation:**

Paper: "Sample P20 was prepared by mixing 80 wt% PLA and 20 wt% PEG."
Recipe: "PLA (80 wt%) and PEG (20 wt%) were mixed."

## 4. Output and final verification

Write one continuous paragraph in the past tense, following the chronological preparation from starting materials through all required precursors to the final material.

Before returning, verify that:

- The correct sample, parent batches, and formulation were used.
- All available preparation stages and material-processing steps were included.
- Every reagent, quantity, operation, and condition is supported by the paper.
- No values were invented, derived, converted, or borrowed from another sample.
- No measured results, appearance descriptions, sample codes, or test procedures remain.

Return **only** the JSON object with the single field `procedure_text`. No headings, explanations, citations, or additional fields.
"""

USER_PROMPT = """
Extract the complete synthesis and preparation procedure for the requested sample, following the system instructions.

Return exactly `{"procedure_text": "..."}` with no additional fields or commentary.

Use the identification record only to identify the target material. Extract all procedural facts exclusively from the supplied paper.

Requested record: paper_uid={paper_uid}; sample_id={sample_id}.

<IDENTIFICATION_RECORD_DATA_ONLY>
{sample_record}
</IDENTIFICATION_RECORD_DATA_ONLY>

<PAPER_MARKDOWN_DATA_ONLY>
{markdown_full_text}
</PAPER_MARKDOWN_DATA_ONLY>
"""

RESPONSE_FORMAT = {'type': 'json_schema', 'json_schema': {'name': 'sample_synthesis_prose', 'strict': True, 'schema': {'type': 'object', 'additionalProperties': False, 'required': ['procedure_text'], 'properties': {'procedure_text': {'type': 'string'}}}}}


## Why the prompt excludes results, labels and test steps

The synthesis paragraph is **predictor input** for property prediction. The model should see only what is known *before* the material is measured: starting materials, amounts, conditions and processing. If the text contains anything that is only known after making and measuring the material, the model is handed part of the answer.

### What leaked in earlier runs (v5–v7, 8 + 7 + 20 samples)

| Type | Example from the runs | Status |
|---|---|---|
| Measured outcome | "91% conversion, 98% selectivity", "mean degree of polymerization of 20", "quantitative deprotection" (P1d polycarbonate) | v5 no-think; v6 no-think still kept 91% and "quantitative" |
| Measured value of an intermediate | "oligomers (Mn = 1265 Da)" (recycled PET); "grafting density 1.3 chains/nm²"; "80 wt% PS / 20 wt% P4VP" (measured composition); "crude precursor (8.77 g, 36.4 mmol)"; "until complete consumption of the monomers" | still present in v6/v7 |
| Appearance | "dense light yellowish liquid", "highly viscous prepolymer", "clear viscous solution", "homogeneous slurry" | v5; mostly removed in v6/v7 |
| Sample label | "End5700", heading "plasticized methacrylate compleximer", "nwLLZO", "regenerated PET", "SiCO" | occasional in v6/v7 |
| Test-only steps | 48 h conditioning at 23 °C/50% RH, Teflon pre-crack film for a fracture test, transfer onto test wafers, tensile-bar mould names | occasional, mostly no-think |

### Why each type harms property prediction

1. **Direct target leakage.** Mn, Mw, dispersity and DP are target properties. "DP 20" or "Mn 1265 Da" in the input gives the answer (or a strong proxy) away.
2. **Indirect leakage through correlated outcomes.** Conversion, yield and "complete consumption" track molecular weight; grafting density and measured composition track Tg, modulus and dispersion; appearance words carry hidden property information ("viscous" → molecular weight, "transparent" → crystallinity, "homogeneous" → better mechanical behaviour, "brittle" → elongation).
3. **Labels encode answers and paper identity.** Many codes contain results: P4-96K (Mn 96 kg/mol), PS80-b-P4VP20¹⁶⁴ (composition and Mn), c-PMMA225 (DP). Series codes (PL05, DM-60) reveal the position in a series. Unique labels let the model recognise the paper and memorise its value range instead of learning chemistry.
4. **Test-only steps add noise or reveal the target.** Conditioning, notching or a crack starter describe how a property was measured, not how the material was made. Identical materials would get different texts, and a test step can reveal which property label goes with the sample.
5. **Gap comments are spurious features.** Phrases like "temperature not reported" depend on writing style, not chemistry, and give the model meaningless patterns to learn.

**Consequence:** the use case is predicting a property *before* synthesis, when there is no yield, no measured Mn, no observed colour and no sample code. A model trained on leaky texts looks accurate on held-out papers (which leak in the same way) and fails on genuinely new recipes. Removing these items is what makes the reported test performance meaningful.

### What must stay

Actual input amounts carried into the next step (the 1.0 g of polymer redissolved, not the 85% yield), formulation compositions and loadings (80 wt% PLA / 20 wt% PEG, not "P20"), stated target or nominal recipe parameters, starting-material specifications, and process endpoints that are stopping criteria of the procedure ("until 95% of the theoretical water was distilled off", "until the agitator power reached 40 W").

### Open cases to settle in the prompt

- **Intermediate identified only by a measured value** (recycled PET: the oligomer is only called "Mn = 1265 Da"): describe it by how it was made and drop the measured value.
- **Endpoint vs outcome:** "until the water was distilled off" stays; "until complete consumption" / "to achieve 91% conversion" goes.
- **Values the authors estimated from measurements** (e.g. volume or mass fractions calculated from SAXS sizes): exclude; keep only stated design/nominal values.


## List samples for a paper

Change `PAPER_UID` and run this cell. Every sample record for that paper is shown.

In [15]:
def get_samples(paper_uid):
    connection = sqlite3.connect(
        f"file:{ROOT / 'data/polymer_samples.db'}?mode=ro", uri=True
    )
    try:
        return pd.read_sql_query(
            "SELECT s.paper_uid, p.canonical_source AS source, s.sample_id, "
            "s.name, s.identifier, s.aliases, s.description "
            "FROM samples s JOIN papers p USING (paper_uid) "
            "WHERE s.paper_uid = ? ORDER BY s.sample_id",
            connection, params=(paper_uid,),
        )
    finally:
        connection.close()

PAPER_UID = "ID000732356"
samples = get_samples(PAPER_UID)
print(f"{PAPER_UID}: {len(samples)} samples")
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(samples)


ID000732356: 16 samples


,paper_uid,source,sample_id,name,identifier,aliases,description
0,ID000732356,springer_nature,1736955,poly1,"Entry 1, Table 1",[],"Polymer synthesized from monomer 1 (3-methyl-1,4-oxathiane-2,5-dione) via anionic ring-opening polymerization using 2-ethylhexyl thioglycolate (5 mol%) as initiator and pyridine (5 mol%) as catalyst in CH2Cl2 at room temperature for 5 days."
1,ID000732356,springer_nature,1736956,poly2,"Entry 2, Table 1",[],"Polymer synthesized from monomer 2 (6-methyl-1,4-oxathiane-2,5-dione) via anionic ring-opening polymerization using 2-ethylhexyl thioglycolate (5 mol%) as initiator and pyridine (5 mol%) as catalyst in CH2Cl2 at room temperature for 5 days."
2,ID000732356,springer_nature,1736957,poly3,"Entry 3, Table 1",[],"Polymer synthesized from monomer 3 (3,6-dimethyl-1,4-oxathiane-2,5-dione) via anionic ring-opening polymerization using 2-ethylhexyl thioglycolate (5 mol%) as initiator and pyridine (5 mol%) as catalyst in CH2Cl2 at room temperature for 5 days."
3,ID000732356,springer_nature,1736958,poly1,"Entries 1–4, Table 2",[],"Polymer synthesized from monomer 1 via anionic ring-opening polymerization using 2-ethylhexyl thioglycolate (1–6 mol%) as initiator and 2,6-lutidine (5–10 mol%) as catalyst in CH2Cl2 at room temperature for 1–5 days."
4,ID000732356,springer_nature,1736959,poly2,"Entries 5–9, Table 2",[],"Polymer synthesized from monomer 2 via anionic ring-opening polymerization using 2-ethylhexyl thioglycolate (1–6 mol%) as initiator and 2,6-lutidine (5–10 mol%) as catalyst in CH2Cl2 at room temperature for 1–4 days."
5,ID000732356,springer_nature,1736960,poly3,"Entry 10, Table 2",[],"Polymer synthesized from monomer 3 via anionic ring-opening polymerization using 2-ethylhexyl thioglycolate (5 mol%) as initiator and 2,6-lutidine (5 mol%) as catalyst in CH2Cl2 at room temperature for 1–3 days."
6,ID000732356,springer_nature,1736961,poly3,"Entries 1–2, Table 3",[],"Polymer synthesized from monomer 3 via anionic ring-opening polymerization using 2-ethylhexyl thioglycolate (1.5–5 mol%) as initiator in 2,6-lutidine as the solvent at room temperature for 1–3 days."
7,ID000732356,springer_nature,1736962,poly1,"Entries 1–3, Table 4",[],"Macrocyclic polymer synthesized from monomer 1 via anionic ring-opening polymerization in 2,6-lutidine as the solvent without a thiol initiator at room temperature for 1–15 days."
8,ID000732356,springer_nature,1736963,poly1,"Entry 4, Table 4",[],"Polymer synthesized from monomer 1 via anionic ring-opening polymerization in a 1:1 (v/v) mixture of 2,6-lutidine and CH2Cl2 as solvent without a thiol initiator at room temperature for 0.25–4 days."
9,ID000732356,springer_nature,1736964,poly2,"Entry 5, Table 4",[],"Polymer synthesized from monomer 2 via anionic ring-opening polymerization in a 1:1 (v/v) mixture of 2,6-lutidine and CH2Cl2 as solvent without a thiol initiator at room temperature for 1–4 days."


## Choose a sample and load the full text

Set both IDs here. This cell reads the selected database record and full local Markdown, then builds `messages`. It does not call Qwen. You can inspect `sample_record`, `paper_text` or `messages` before sending.

In [18]:
PAPER_UID = "ID000732356"
SAMPLE_ID = 1736958

# Fetch again so the chosen UID need not match the preceding display cell.
paper_samples = get_samples(PAPER_UID)
selected = paper_samples.loc[paper_samples.sample_id == SAMPLE_ID]
if len(selected) != 1:
    raise ValueError(f"Sample {SAMPLE_ID} does not belong to paper {PAPER_UID}.")

row = selected.iloc[0]
md_path = ROOT / "data/fulltext" / row["source"] / PAPER_UID / f"{PAPER_UID}.md"
paper_text = md_path.read_text(encoding="utf-8")
if not paper_text.strip():
    raise ValueError(f"The paper Markdown is empty: {md_path}")
sample_record = {key: row[key] for key in ("name", "identifier", "aliases", "description")}

values = {
    "paper_uid": PAPER_UID,
    "sample_id": str(SAMPLE_ID),
    "sample_record": json.dumps(sample_record, ensure_ascii=False),
    "markdown_full_text": paper_text,
}
user_message = re.sub(
    r"\{(paper_uid|sample_id|sample_record|markdown_full_text)\}",
    lambda match: values[match.group(1)], USER_PROMPT,
)
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": user_message},
]
display(selected)
print(f"Loaded full Markdown: {len(paper_text):,} characters")


,paper_uid,source,sample_id,name,identifier,aliases,description
3,ID000732356,springer_nature,1736958,poly1,"Entries 1–4, Table 2",[],Polymer synthesized from monomer 1 via anionic...


Loaded full Markdown: 36,620 characters


## Send to Qwen and display the response

Each execution makes one model request. The full response remains available as `response`; the paragraph is `procedure_text`. Nothing is written to disk.

In [19]:
import textwrap

async with AsyncOpenAI(
    base_url=BASE_URL, api_key="none", timeout=1200, max_retries=0
) as client:
    response = await client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=0.0,
        max_tokens=20_000,
        extra_body={"chat_template_kwargs": {"enable_thinking": THINKING}},
        response_format=RESPONSE_FORMAT,
    )

answer = response.choices[0].message.content or ""
procedure_text = None
try:
    parsed = json.loads(answer)
    if isinstance(parsed, dict) and isinstance(parsed.get("procedure_text"), str):
        procedure_text = parsed["procedure_text"]
except json.JSONDecodeError:
    pass

if procedure_text is None:
    print(answer or "[No answer content returned]")
else:
    print(textwrap.fill(procedure_text or "[Empty synthesis returned]", width=100))
if response.choices[0].finish_reason != "stop":
    print("\nResponse may be incomplete. Finish reason:", response.choices[0].finish_reason)


Thiolactic acid (8.7 mL, 0.1 mol) was slowly added via a syringe to bromoacetyl bromide (8.8 mL, 0.1
mol) in a two-neck flask under N2. The mixture was stirred at room temperature overnight, while the
evolved HBr gas was introduced over aqueous NaOH through an outlet natural rubber tube with an N2
stream. Water (200 mL) was added to the reaction mixture, which was subsequently extracted with
CH2Cl2 (2 × 200 mL). The combined organic layer was washed with brine, dried over MgSO4, and
concentrated in vacuo. The resulting residue (16.46 g, 72.4 mmol) was dissolved in dry MeCN (200 mL)
under N2 and cooled to 0 °C. Dry (iPr)2NEt (12 mL, 70.6 mmol) dissolved in dry MeCN (17 mL) was
added via a syringe. The mixture was allowed to warm to room temperature and stirred overnight.
Diethyl ether was added, and the (iPr)2NEt hydrobromide was removed by filtration. The filtrate was
evaporated under reduced pressure, and the crude mixture was purified by silica gel column
chromatography using CH2Cl2.

In [ ]:
"""
Thiolactic acid (8.7 mL, 0.1 mol) was slowly added via a syringe to bromoacetyl bromide (8.8 mL, 0.1
mol) in a two-neck flask under N2. The mixture was stirred at room temperature overnight, while the
evolved HBr gas was introduced over aqueous NaOH through an outlet natural rubber tube with an N2
stream. Water (200 mL) was added to the reaction mixture, which was subsequently extracted with
CH2Cl2 (2 × 200 mL). The combined organic layer was washed with brine, dried over MgSO4, and
concentrated in vacuo. The resulting residue (16.46 g, 72.4 mmol) was dissolved in dry MeCN (200 mL)
under N2 and cooled to 0 °C. Dry (iPr)2NEt (12 mL, 70.6 mmol) dissolved in dry MeCN (17 mL) was
added via a syringe. The mixture was allowed to warm to room temperature and stirred overnight.
Diethyl ether was added, and the (iPr)2NEt hydrobromide precipitate was removed by filtration. The
filtrate was evaporated under reduced pressure, and the crude mixture was purified by silica gel
column chromatography using CH2Cl2. The purified product was distilled at 131 °C under 270 Pa and
crystallized in a refrigerator. The synthesized monomer (100 mg, 0.684 mmol) was dissolved in dry
CH2Cl2 (0.46 mL) in a test tube under N2. A solution of dry 2-ethylhexyl thioglycolate (140 µL, 136
mg, 0.666 mmol) in dry CH2Cl2 (2.0 mL) was prepared in advance, and a portion (147 mg, 0.0342 mmol)
was added to the test tube. A solution of dry pyridine in dry CH2Cl2 was also prepared in advance,
and a portion corresponding to 5 mol% (0.0342 mmol) was subsequently added to the test tube. The
mixture was stirred under a closed system at room temperature for 5 days. Amberlyst 15E (15 mg) was
added to the reaction mixture to exclude pyridine. After 30 min of stirring, the supernatant was
collected by filtration and dried under vacuum. The polymer was isolated by reprecipitation into
diethyl ether.
"""